# Land-Cover Change Detection

This notebook analyses land-cover changes in Lublin between 2006, 2016 and 2026 using the final Random Forest classification.

The workflow:
- reconstructs the classified land-cover maps for all three analysed years,
- encodes class-to-class transitions,
- calculates the area of each transition,
- identifies transitions associated with urban growth,
- calculates newly classified built-up areas,
- calculates vegetation-to-non-vegetation transitions,
- visualizes the spatial distribution of the main changes,
- summarizes the results for the analysed periods.

The analysed periods are:
- 2006–2016,
- 2016–2026,
- 2006–2026.

## Environment and project setup

The required Python libraries and reusable Earth Engine helper functions are loaded. Project paths are defined relative to the repository root.

In [29]:
from pathlib import Path
import sys
import json

import ee
import geemap
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from gee_utils import (
    preprocess_landsat5,
    preprocess_landsat8,
    create_composite,
    add_indices,
    split_class,
    sample_year
)

ee.Initialize()

print("Earth Engine initialized")

Earth Engine initialized


## Study area

The administrative boundary of Lublin is loaded and converted to WGS84. The resulting geometry defines the common Area of Interest used for the change-detection analysis.

In [30]:
CITY_PATH = PROJECT_ROOT / "data" / "boundaries" / "city" / "lublin_city_boundary.shp"

city = gpd.read_file(CITY_PATH).to_crs(epsg=4326)

city_geojson = json.loads(city.to_json())
aoi = ee.Geometry(city_geojson["features"][0]["geometry"])

print("AOI ready")

AOI ready


## Reconstruct the predictor composites

The same Landsat preprocessing, summer observation window and spectral-index calculation used in the previous notebooks are applied here.  
This ensures that the classifications used for change detection are based on exactly the same predictor structure for all analysed years.

In [31]:
composite_2006 = create_composite(
    "LANDSAT/LT05/C02/T1_L2",
    "2006-06-01",
    "2006-09-01",
    aoi,
    preprocess_landsat5
)

composite_2016 = create_composite(
    "LANDSAT/LC08/C02/T1_L2",
    "2016-06-01",
    "2016-09-01",
    aoi,
    preprocess_landsat8
)

composite_2026 = create_composite(
    "LANDSAT/LC08/C02/T1_L2",
    "2026-06-01",
    "2026-09-01",
    aoi,
    preprocess_landsat8
)

composite_2006_idx = add_indices(composite_2006)
composite_2016_idx = add_indices(composite_2016)
composite_2026_idx = add_indices(composite_2026)

predictor_bands = [
    "blue", "green", "red",
    "nir", "swir1", "swir2",
    "NDVI", "NDBI"
]

print("Predictor composites ready")

Predictor composites ready


## Reconstruct the final Random Forest model

The saved training polygons prepared in the classification notebook are loaded from `data/training/`.  
The same training data, class definitions and model parameters are reused to reconstruct the final Random Forest classifier.

In [32]:
TRAINING_DIR = PROJECT_ROOT / "data" / "training"

vegetation_gdf = gpd.read_file(TRAINING_DIR / "vegetation.geojson")
built_up_gdf   = gpd.read_file(TRAINING_DIR / "built_up.geojson")
other_gdf      = gpd.read_file(TRAINING_DIR / "other.geojson")
water_gdf      = gpd.read_file(TRAINING_DIR / "water.geojson")

training_gdf = gpd.GeoDataFrame(
    pd.concat(
        [
            vegetation_gdf,
            built_up_gdf,
            other_gdf,
            water_gdf
        ],
        ignore_index=True
    ),
    crs=vegetation_gdf.crs
)

training_gdf = training_gdf.to_crs(epsg=4326)

training_fc = geemap.geopandas_to_ee(training_gdf)

print(training_gdf["class_name"].value_counts())
print("Total training polygons:", len(training_gdf))

class_name
vegetation    18
water         13
built_up      12
other          9
Name: count, dtype: int64
Total training polygons: 52


### Reproduce the training subset

The same class-stratified polygon split used in the classification notebook is reproduced using fixed random seeds.  
Only the training polygons are required here because model validation was performed in the previous notebook.

In [33]:
train_0, test_0 = split_class(training_fc, 0, 5, 42)
train_1, test_1 = split_class(training_fc, 1, 4, 43)
train_2, test_2 = split_class(training_fc, 2, 3, 44)
train_3, test_3 = split_class(training_fc, 3, 4, 45)

train_polygons = (
    train_0
    .merge(train_1)
    .merge(train_2)
    .merge(train_3)
)

print(
    "Train polygons:",
    train_polygons.aggregate_histogram("class").getInfo()
)

Train polygons: {'0': 13, '1': 8, '2': 6, '3': 9}


### Extract the multi-year training sample

Predictor values are extracted from the training polygons for 2006, 2016 and 2026 and merged into one multi-year training sample.

In [34]:
train_2006 = sample_year(
    composite_2006_idx,
    train_polygons,
    predictor_bands
)

train_2016 = sample_year(
    composite_2016_idx,
    train_polygons,
    predictor_bands
)

train_2026 = sample_year(
    composite_2026_idx,
    train_polygons,
    predictor_bands
)

train_samples = (
    train_2006
    .merge(train_2016)
    .merge(train_2026)
)

print("Number of training pixels:", train_samples.size().getInfo())

Number of training pixels: 2202


### Train the final classifier

A Random Forest model with 200 trees is trained using the combined multi-year training sample.  
The same classifier is subsequently applied to all three analysed years.

In [35]:
rf_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=200,
    seed=42
).train(
    features=train_samples,
    classProperty="class",
    inputProperties=predictor_bands
)

print("Random Forest model ready")

Random Forest model ready


### Final classified maps

The reconstructed Random Forest model is applied to the predictor composites for 2006, 2016 and 2026.

The resulting maps use the same four classes:

- `0` — vegetation,
- `1` — built-up,
- `2` — other,
- `3` — water.e,
- `3` — woda.

In [36]:
rf_2006 = (
    composite_2006_idx
    .select(predictor_bands)
    .classify(rf_classifier)
    .rename("class")
    .clip(aoi)
)

rf_2016 = (
    composite_2016_idx
    .select(predictor_bands)
    .classify(rf_classifier)
    .rename("class")
    .clip(aoi)
)

rf_2026 = (
    composite_2026_idx
    .select(predictor_bands)
    .classify(rf_classifier)
    .rename("class")
    .clip(aoi)
)

print("Classifications ready")

Classifications ready


## Land-cover transition analysis

Class transitions are encoded using:

`transition = old_class × 10 + new_class`

For example:

- `01` represents vegetation → built-up,
- `10` represents built-up → vegetation,
- `11` represents built-up → built-up.

This encoding allows every class-to-class transition to be identified and quantified.

In [37]:
change_2006_2016 = rf_2006.multiply(10).add(rf_2016).rename("transition")
change_2016_2026 = rf_2016.multiply(10).add(rf_2026).rename("transition")
change_2006_2026 = rf_2006.multiply(10).add(rf_2026).rename("transition")

print("Change layers ready")

Change layers ready


### Transition-area statistics

Pixel area is aggregated for each transition code and converted to square kilometres.  
The numerical transition codes are then translated into readable source and destination class names.

In [38]:
def transition_areas(change_image, aoi):
    area_image = ee.Image.pixelArea().addBands(change_image)

    stats = area_image.reduceRegion(
        reducer=ee.Reducer.sum().group(
            groupField=1,
            groupName="transition"
        ),
        geometry=aoi,
        scale=30,
        maxPixels=1e9
    )

    return stats

In [39]:
stats_2006_2016 = transition_areas(change_2006_2016, aoi).getInfo()
stats_2016_2026 = transition_areas(change_2016_2026, aoi).getInfo()
stats_2006_2026 = transition_areas(change_2006_2026, aoi).getInfo()

In [40]:
class_names = {
    0: "vegetation",
    1: "built_up",
    2: "other",
    3: "water"
}

def transition_table(stats, period):
    rows = []

    for item in stats["groups"]:
        code = int(item["transition"])

        old_class = code // 10
        new_class = code % 10

        rows.append({
            "period": period,
            "from_class": class_names[old_class],
            "to_class": class_names[new_class],
            "area_km2": item["sum"] / 1_000_000
        })

    return pd.DataFrame(rows).sort_values("area_km2", ascending=False)

### Transition matrices

Transition tables are produced for:

- 2006–2016,
- 2016–2026,
- 2006–2026.

The tables contain both stable land-cover classes and detected class changes.

In [41]:
table_2006_2016 = transition_table(
    stats_2006_2016,
    "2006-2016"
)

table_2016_2026 = transition_table(
    stats_2016_2026,
    "2016-2026"
)

table_2006_2026 = transition_table(
    stats_2006_2026,
    "2006-2026"
)

In [42]:
table_2006_2016

,period,from_class,to_class,area_km2
5,2006-2016,built_up,built_up,41.787681
0,2006-2016,vegetation,vegetation,40.516818
8,2006-2016,other,vegetation,22.395012
10,2006-2016,other,other,12.660092
9,2006-2016,other,built_up,7.524500
4,2006-2016,built_up,vegetation,6.862790
2,2006-2016,vegetation,other,4.314213
1,2006-2016,vegetation,built_up,3.621595
15,2006-2016,water,water,3.109640
6,2006-2016,built_up,other,3.019681


In [43]:
table_2016_2026

,period,from_class,to_class,area_km2
0,2016-2026,vegetation,vegetation,48.621008
5,2016-2026,built_up,built_up,44.055960
2,2016-2026,vegetation,other,12.364849
1,2016-2026,vegetation,built_up,9.400333
10,2016-2026,other,other,8.709622
9,2016-2026,other,built_up,6.344451
8,2016-2026,other,vegetation,4.889258
6,2016-2026,built_up,other,4.499743
4,2016-2026,built_up,vegetation,3.467220
15,2016-2026,water,water,3.111919


In [44]:
table_2006_2026

,period,from_class,to_class,area_km2
5,2006-2026,built_up,built_up,40.008317
0,2006-2026,vegetation,vegetation,35.098716
10,2006-2026,other,other,15.554817
8,2006-2026,other,vegetation,14.292138
9,2006-2026,other,built_up,12.539396
1,2006-2026,vegetation,built_up,7.088865
4,2006-2026,built_up,vegetation,6.567373
2,2006-2026,vegetation,other,5.925776
6,2006-2026,built_up,other,4.014234
15,2006-2026,water,water,3.063376


## Urban-growth-related transitions

A subset of transitions directly related to changes in built-up land is extracted for easier interpretation.

The selected transitions are:

- vegetation → built-up,
- other → built-up,
- built-up → vegetation,
- built-up → other.

In [45]:
def important_transitions(table):
    return table[
        (
            (table["from_class"] == "vegetation") &
            (table["to_class"] == "built_up")
        )
        |
        (
            (table["from_class"] == "other") &
            (table["to_class"] == "built_up")
        )
        |
        (
            (table["from_class"] == "built_up") &
            (table["to_class"] == "vegetation")
        )
        |
        (
            (table["from_class"] == "built_up") &
            (table["to_class"] == "other")
        )
    ].reset_index(drop=True)

In [46]:
important_2006_2016 = important_transitions(table_2006_2016)
important_2016_2026 = important_transitions(table_2016_2026)
important_2006_2026 = important_transitions(table_2006_2026)

important_changes = pd.concat(
    [
        important_2006_2016,
        important_2016_2026,
        important_2006_2026
    ],
    ignore_index=True
)

important_changes

,period,from_class,to_class,area_km2
0,2006-2016,other,built_up,7.524500
1,2006-2016,built_up,vegetation,6.862790
2,2006-2016,vegetation,built_up,3.621595
3,2006-2016,built_up,other,3.019681
4,2016-2026,vegetation,built_up,9.400333
5,2016-2026,other,built_up,6.344451
6,2016-2026,built_up,other,4.499743
7,2016-2026,built_up,vegetation,3.467220
8,2006-2026,other,built_up,12.539396
9,2006-2026,vegetation,built_up,7.088865


## Newly classified built-up areas

New built-up areas are defined as pixels that were not classified as built-up at the beginning of a period but were classified as built-up at the end of that period.  
This indicator combines transitions from vegetation, other surfaces and water into the built-up class.

In [47]:
def new_built_up(old_classification, new_classification):
    return (
        old_classification.neq(1)
        .And(new_classification.eq(1))
        .selfMask()
        .rename("new_built_up")
    )

In [48]:
new_built_2006_2016 = new_built_up(rf_2006, rf_2016)
new_built_2016_2026 = new_built_up(rf_2016, rf_2026)
new_built_2006_2026 = new_built_up(rf_2006, rf_2026)

In [49]:
def masked_area_km2(mask_image, aoi):
    area = (
        ee.Image.pixelArea()
        .updateMask(mask_image)
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=aoi,
            scale=30,
            maxPixels=1e9
        )
    )

    return ee.Number(area.get("area")).divide(1_000_000)

### New built-up results

The detected newly classified built-up area is approximately:

- 11.32 km² for 2006–2016,
- 15.85 km² for 2016–2026,
- 19.90 km² for 2006–2026.

The 2006–2026 result is not the sum of the two shorter periods. It compares only the initial 2006 classification with the final 2026 classification.

In [50]:
print(
    "New built-up 2006-2016:",
    masked_area_km2(new_built_2006_2016, aoi).getInfo(),
    "km²"
)

print(
    "New built-up 2016-2026:",
    masked_area_km2(new_built_2016_2026, aoi).getInfo(),
    "km²"
)

print(
    "New built-up 2006-2026:",
    masked_area_km2(new_built_2006_2026, aoi).getInfo(),
    "km²"
)

New built-up 2006-2016: 11.323698347712742 km²
New built-up 2016-2026: 15.85314359306521 km²
New built-up 2006-2026: 19.900785961362093 km²


### Spatial distribution of newly classified built-up areas

The detected newly classified built-up pixels are displayed for all three analysed periods.  
The maps are used to examine the spatial pattern of urban expansion rather than to interpret every individual 30 m pixel as a newly constructed building.

In [51]:
Map_change = geemap.Map()
Map_change.centerObject(aoi, 11)

Map_change.addLayer(
    new_built_2006_2016,
    {"palette": ["FF8C00"]},
    "New built-up 2006-2016",
    False
)

Map_change.addLayer(
    new_built_2016_2026,
    {"palette": ["DC143C"]},
    "New built-up 2016-2026",
    False
)

Map_change.addLayer(
    new_built_2006_2026,
    {"palette": ["8B0000"]},
    "New built-up 2006-2026",
    True
)

Map_change

Map(center=[51.225732957054035, 22.556598958902953], controls=(WidgetControl(options=['position', 'transparent…

## Vegetation-to-non-vegetation transitions

Vegetation loss is defined here as pixels classified as vegetation at the beginning of a period and as any non-vegetation class at the end of the period.  
This indicator should be interpreted as a classification transition rather than as a direct measurement of permanent green-space removal.

In [52]:
def vegetation_loss(old_classification, new_classification):
    return (
        old_classification.eq(0)
        .And(new_classification.neq(0))
        .selfMask()
        .rename("vegetation_loss")
    )

In [53]:
vegetation_loss_2006_2016 = vegetation_loss(rf_2006, rf_2016)
vegetation_loss_2016_2026 = vegetation_loss(rf_2016, rf_2026)
vegetation_loss_2006_2026 = vegetation_loss(rf_2006, rf_2026)

### Vegetation-loss results

The detected vegetation-to-non-vegetation area is approximately:

- 8.00 km² for 2006–2016,
- 22.34 km² for 2016–2026,
- 13.41 km² for 2006–2026.

The relatively large value for 2016–2026 should be interpreted cautiously because the `vegetation → other` transition can also reflect agricultural conditions, dry surfaces, seasonal differences and classification uncertainty.

In [54]:
print(
    "Vegetation loss 2006-2016:",
    masked_area_km2(vegetation_loss_2006_2016, aoi).getInfo(),
    "km²"
)

print(
    "Vegetation loss 2016-2026:",
    masked_area_km2(vegetation_loss_2016_2026, aoi).getInfo(),
    "km²"
)

print(
    "Vegetation loss 2006-2026:",
    masked_area_km2(vegetation_loss_2006_2026, aoi).getInfo(),
    "km²"
)

Vegetation loss 2006-2016: 7.996238561555516 km²
Vegetation loss 2016-2026: 22.33842840787641 km²
Vegetation loss 2006-2026: 13.4143409435262 km²


### Spatial distribution of vegetation-to-non-vegetation transitions

The detected vegetation-to-non-vegetation transitions are displayed for the three analysed periods to examine their spatial distribution across Lublin.

In [55]:
Map_veg_loss = geemap.Map()
Map_veg_loss.centerObject(aoi, 11)

Map_veg_loss.addLayer(
    vegetation_loss_2006_2016,
    {"palette": ["FFA500"]},
    "Vegetation loss 2006-2016",
    False
)

Map_veg_loss.addLayer(
    vegetation_loss_2016_2026,
    {"palette": ["FF1493"]},
    "Vegetation loss 2016-2026",
    False
)

Map_veg_loss.addLayer(
    vegetation_loss_2006_2026,
    {"palette": ["8B008B"]},
    "Vegetation loss 2006-2026",
    True
)

Map_veg_loss

Map(center=[51.225732957054035, 22.556598958902953], controls=(WidgetControl(options=['position', 'transparent…

## Change summary

The main change indicators are combined into one summary table for direct comparison between the analysed periods.

In [56]:
summary_changes = pd.DataFrame({
    "period": [
        "2006-2016",
        "2016-2026",
        "2006-2026"
    ],
    "new_built_up_km2": [
        masked_area_km2(new_built_2006_2016, aoi).getInfo(),
        masked_area_km2(new_built_2016_2026, aoi).getInfo(),
        masked_area_km2(new_built_2006_2026, aoi).getInfo()
    ],
    "vegetation_loss_km2": [
        masked_area_km2(vegetation_loss_2006_2016, aoi).getInfo(),
        masked_area_km2(vegetation_loss_2016_2026, aoi).getInfo(),
        masked_area_km2(vegetation_loss_2006_2026, aoi).getInfo()
    ]
})

summary_changes

,period,new_built_up_km2,vegetation_loss_km2
0,2006-2016,11.323698,7.996239
1,2016-2026,15.853144,22.338428
2,2006-2026,19.900786,13.414341


## Interpretation and limitations

The results indicate a larger amount of newly classified built-up land in 2016–2026 than in 2006–2016.

However, the detected transitions represent differences between classified Landsat images rather than exact cadastral measurements of land-use conversion.

The main sources of uncertainty include:
- the 30 m spatial resolution of Landsat,
- spectral mixing within individual pixels,
- classification errors,
- differences between Landsat 5 and Landsat 8,
- seasonal and agricultural variability,
- confusion between bare or dry surfaces and built-up land.

For this reason, the strongest evidence of urban expansion is provided by spatially coherent transitions into the built-up class rather than by isolated changed pixels.